In [0]:
df = spark.read.csv("/Volumes/workspace/default/data_upload/netflix_titles.csv", header="true", inferSchema="true")

df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [0]:
#csv was being misread as a string so needed to reinforce the schema
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .csv("/Volumes/workspace/default/data_upload/netflix_titles.csv")

In [0]:
#test the row again
df.filter(df["show_id"] == "s74").show(truncate=False)

+-------+-----+------------+------------+------------------------------------------------------------------------------------------------------------------------------------------+-------+------------------+------------+------+--------+----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|show_id|type |title       |director    |cast                                                                                                                                      |country|date_added        |release_year|rating|duration|listed_in                   |description                                                                                                                                            |
+-------+-----+------------+------------+---------------------------------------------------------------------------------------------------------------------------

In [0]:
#1. Count of Movies vs TV Shows.
df.groupBy("type").count().show()

+-------+-----+
|   type|count|
+-------+-----+
|TV Show| 2676|
|  Movie| 6131|
+-------+-----+



In [0]:
#2. Count of null/missing `director` values.
df.filter(df["director"].isNull()).count()

2634

In [0]:
# 3. Count of titles per `rating`, sorted descending.
df.filter(df["rating"].isNotNull()).groupBy("rating").count().orderBy("count", ascending=False)

DataFrame[rating: string, count: bigint]

In [0]:
# 4. Top 10 `listed_in` genres by title count
df.filter(df["listed_in"].isNotNull()).groupBy("listed_in").count().orderBy("count", ascending=False).limit(10).show()

+--------------------+-----+
|           listed_in|count|
+--------------------+-----+
|Dramas, Internati...|  362|
|       Documentaries|  359|
|     Stand-Up Comedy|  334|
|Comedies, Dramas,...|  274|
|Dramas, Independe...|  252|
|            Kids' TV|  220|
|Children & Family...|  215|
|Children & Family...|  201|
|Documentaries, In...|  186|
|Dramas, Internati...|  180|
+--------------------+-----+



In [0]:
# 5. Top 15 genres by title count
df.filter(df["listed_in"].isNotNull()).groupBy("listed_in").count().orderBy("count", ascending=False).limit(15).show()

+--------------------+-----+
|           listed_in|count|
+--------------------+-----+
|Dramas, Internati...|  362|
|       Documentaries|  359|
|     Stand-Up Comedy|  334|
|Comedies, Dramas,...|  274|
|Dramas, Independe...|  252|
|            Kids' TV|  220|
|Children & Family...|  215|
|Children & Family...|  201|
|Documentaries, In...|  186|
|Dramas, Internati...|  180|
|Comedies, Interna...|  176|
|Comedies, Interna...|  152|
|              Dramas|  138|
|Dramas, Internati...|  134|
|Action & Adventur...|  132|
+--------------------+-----+



In [0]:
# 6. Type of available content
df.select("type").distinct().show()

+-------+
|   type|
+-------+
|TV Show|
|  Movie|
+-------+



In [0]:
# 7. Quantity of newly added contents by month and year.
from pyspark.sql.functions import to_date
from pyspark.sql.functions import year, month
from pyspark.sql.functions import trim, to_date

df_dates = df.withColumn(
    "date_added",
    to_date(trim("date_added"), "MMMM d, yyyy")
)

display(
    df_dates.filter(df_dates["date_added"].isNotNull())
        .groupBy(
            year("date_added").alias("year"),
            month("date_added").alias("month")
        )
        .count()
        .orderBy("year", "month")
)

year,month,count
2008,1,1
2008,2,1
2009,5,1
2009,11,1
2010,11,1
2011,5,1
2011,9,1
2011,10,11
2012,2,1
2012,11,1


In [0]:
# 8. Average duration of Tv Shows in seasons
from pyspark.sql.functions import split, avg

tv_shows = df.filter(df["type"] == "TV Show")

tv_shows = tv_shows.withColumn("seasons", split("duration", " ")[0].cast("int"))

tv_shows.select(avg("seasons")).show()

+-----------------+
|     avg(seasons)|
+-----------------+
|1.764947683109118|
+-----------------+



In [0]:
# 9.Its Christmas! What movies/Tv shows are you binge watching this weekend?
df.filter(df["title"].contains("Christmas")).show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+-----------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|       date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+-----------------+------------+------+---------+--------------------+--------------------+
|  s1517|TV Show|  Home for Christmas|                NULL|Ida Elise Broch, ...|              Norway|December 18, 2020|        2020| TV-MA|2 Seasons|International TV ...|Tired of the cons...|
|  s1524|  Movie|An Unremarkable C...|  Juan Camilo Pinzon|Antonio Sanint, L...|            Colombia|December 17, 2020|        2020| TV-14|   83 min|Comedies, Dramas,...|An accountant and...|
|  s1536|TV Show|How To Ruin Chris...|  

In [0]:
# 10. My friend is a Ryan Reynold fan , I am thinking of making a movie compilation. What movies should I include ?
df.filter(df["cast"].contains("Ryan Reynolds")).show()

+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+
|show_id| type|               title|            director|                cast|             country|        date_added|release_year|rating|duration|           listed_in|         description|
+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+
|    s47|Movie|          Safe House|     Daniel Espinosa|Denzel Washington...|South Africa, Uni...|September 16, 2021|        2012|     R| 115 min|  Action & Adventure|Young CIA operati...|
|   s144|Movie|       Green Lantern|     Martin Campbell|Ryan Reynolds, Bl...|       United States| September 1, 2021|        2011| PG-13| 114 min|Action & Adventur...|Test pilot Hal Jo...|
|  s3032|Movie|Betty White: Firs...|     Steve Boe

In [0]:
# 11. How many movies or shows were released during COVID season.
covid_time_window = df.filter((df["release_year"] >= 2020) & (df["release_year"] <= 2021))
covid_time_window.groupBy("type").count().show()


+-------+-----+
|   type|count|
+-------+-----+
|TV Show|  751|
|  Movie|  794|
+-------+-----+



In [0]:
# 12. I am on a short flight of 2 hours, which movies or shows I should download to watch on the flight ?
movies = df.filter(df["type"] == "Movie")
movies = movies.withColumn("minutes", split("duration", " ")[0].cast("int"))
movies.filter(
    (movies["minutes"] > 89) & 
    (movies["minutes"] < 121)
).show()

+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+-------+
|show_id| type|               title|            director|                cast|             country|        date_added|release_year|rating|duration|           listed_in|         description|minutes|
+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+-------+
|     s1|Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|  90 min|       Documentaries|As her father nea...|     90|
|     s7|Movie|My Little Pony: A...|Robert Cullen, Jo...|Vanessa Hudgens, ...|                NULL|September 24, 2021|        2021|    PG|  91 min|Children & Family...|Equestria's divid...|     91|
|    s10|M